In [1]:
#!pip install faiss-cpu


In [2]:
import os
import requests
import logging
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_community.vectorstores import FAISS
import faiss
import numpy as np


api_key = os.getenv("AICREDITS_API_KEY")
base_url = os.getenv("AICREDITS_BASE_URL")


C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_14296\766622890.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### define tools

In [3]:
import json

@tool
def query_wolfram_alpha(expression: str) -> str:
    """Query Wolfram Alpha to compute expressions or retrieve information.
    
    Args: 
        expression (str): The mathematical expression or query to evaluate.
    Returns: 
        str: The result of the computation or the retrieved information.
    """
    print(f" [MOCK WOLFRAM] Evaluating expression: '{expression}'")
    
    # Optional: Basic local evaluator if the model passes clean math strings
    try:
        # Cleans common wording out if the model asks a conversational math question
        clean_expr = expression.lower().replace("what is", "").replace("?", "").strip()
        # Handle a few common textbook string formulas safely
        if "^" in clean_expr: 
            clean_expr = clean_expr.replace("^", "**")
            
        # Safely evaluate simple math strings locally
        calc_result = eval(clean_expr, {"__builtins__": None}, {})
        return f"Result: {calc_result}"
    except Exception:
        # Clever realistic fallback answer for complex queries
        return f"Wolfram Alpha computed result for query '{expression}': operation completed successfully."


@tool
def trigger_zapier_webhook(zap_id: str, payload: dict) -> str:
    """Trigger a Zapier webhook to execute a predefined Zap.
    
    Args:
        zap_id (str): The unique identifier for the Zap to be triggered.
        payload (dict): The data to send to the Zapier webhook.
    Returns:
        str: Confirmation message upon successful triggering of the Zap.
    """
    print(f" [MOCK ZAPIER] Webhook intercept triggered!")
    print(f"   ↳ Zap ID: {zap_id}")
    print(f"   ↳ Payload Data Sent: {json.dumps(payload, indent=2)}")
    
    return f"Zapier webhook '{zap_id}' successfully triggered (Mock Mode)."


@tool
def send_slack_message(channel: str, message: str) -> str:
    """Send a message to a specified Slack channel.
    
    Args:
        channel (str): The Slack channel ID or name where the message will be sent.
        message (str): The content of the message to send.
    Returns:
        str: Confirmation message upon successful sending of the Slack message.
    """
    print(f" [MOCK SLACK] Outbound Message Dispatched:")
    print(f"   ↳ Destination Channel: {channel}")
    print(f"   ↳ Text Body: \"{message}\"")
    
    return f"Message successfully sent to Slack Channel '{channel}'."


In [4]:
llm = ChatOpenAI(model_name= "gpt-4o", api_key=api_key, base_url=base_url)

### initialize the OpenAI Embeddings

In [5]:
#initialize the OpenAI Embeddings
embeddings = OpenAIEmbeddings(openai_api_key = api_key, base_url=base_url)

In [6]:
#Tool Description
tool_descriptions = {
    "query_wolfram_alpha": '''Use Wolfram Alpha to compute mathematical expressions or retrieve information.''',
    "trigger_zapier_webhook": '''Trigger a Zapier webhook to execute predefined automated workflows.''',
    "send_slack_message": '''Send messages to specific Slack channels to communicate with team members.'''
}

### Create embeddings for each tool description

In [7]:
tool_embeddings = []
tool_names = []

for tool_name, description in tool_descriptions.items():
    embedding = embeddings.embed_query(description)
    tool_embeddings.append(embedding)
    tool_names.append(tool_name)

### Initialize FAISS vector store

In [8]:
dimension = len(tool_embeddings[0])
index = faiss.IndexFlatL2(dimension)

### Normalize for cosine similarity

In [9]:
faiss.normalize_L2(np.array(tool_embeddings).astype('float32'))

### convert list to FAISS-compatible format

In [10]:
tool_embeddings_np = np.array(tool_embeddings).astype('float32')
index.add(tool_embeddings_np)

### Map index to tool functions

In [11]:
index_to_tool = {
    0: query_wolfram_alpha,
    1: trigger_zapier_webhook,
    2: send_slack_message
}


In [12]:
def select_tool(query: str, top_k: int =1) -> list:
    """
    Select the most relevant tool(s) based on the user's query using
    vector-based retrieval.
    Args:
    query (str): The user's input query.
    top_k (int): Number of top tools to retrieve.
    Returns:
    list: List of selected tool functions.
    """
    query_embedding = np.array(embeddings.embed_query(query)).astype('float32')
    faiss.normalize_L2(query_embedding.reshape(1,-1))
    D, I = index.search(query_embedding.reshape(1,-1), top_k)
    selected_tools = [index_to_tool[idx] for idx in I[0] if idx in index_to_tool]
    return selected_tools

In [13]:
def determine_parameters(query: str, tool_name: str) -> dict:
    """
    Use the LLM to analyze the query and determine the parameters for the tool
    to be invoked.
    Args:
    query (str): The user's input query.
    tool_name (str): The selected tool name.
    Returns:
    dict: Parameters for the tool.
    """
    messages = [
        HumanMessage(content=f"""Based on the user's query: {query}, what parameters 
        should be used for the tool '{tool_name}'""")
    ]

    ### Call the LLM to extract Parameters
    response = llm.invoke(messages)

    ai_text = response.content

     ### Logic to parse response from LLM
    parameters = {}
    
    if tool_name == "query_wolfram_alpha":
        # Pass the clean user query string straight to your mock tool
        parameters["expression"] = query
    elif tool_name == "trigger_zapier_webhook":
        parameters["zap_id"] = "123456"
        parameters["payload"] = {"data": query}
        
    elif tool_name == "send_slack_message":
        parameters["channel"] = "#general"
        parameters["message"] = query
    
    return parameters
    

In [14]:
#example query
user_query = "Solve this equation: 2x + 3 = 7"

In [15]:
###select the top tool
selected_tools = select_tool(user_query, top_k=1)
tool_name = selected_tools[0] if selected_tools else None

In [16]:
tool_name

StructuredTool(name='query_wolfram_alpha', description='Query Wolfram Alpha to compute expressions or retrieve information.\n\n    Args: \n        expression (str): The mathematical expression or query to evaluate.\n    Returns: \n        str: The result of the computation or the retrieved information.', args_schema=<class 'langchain_core.utils.pydantic.query_wolfram_alpha'>, func=<function query_wolfram_alpha at 0x000002018E04FA60>)

In [17]:
if tool_name:
    tool_string_name = tool_name.name
    args = determine_parameters(user_query, tool_string_name)
    
    # invoke selected tool
    try:
        tool_result = tool_name.invoke(args)
        print(f"Tool '{tool_string_name}' Result: {tool_result}")
    except ValueError as e:
        print(f"Error Invoking tool '{tool_string_name}': {e}")
else:
    print("No tool was selected")


 [MOCK WOLFRAM] Evaluating expression: 'Solve this equation: 2x + 3 = 7'
Tool 'query_wolfram_alpha' Result: Wolfram Alpha computed result for query 'Solve this equation: 2x + 3 = 7': operation completed successfully.
